# 10 — Download + verify (CPU). Scratch = /tmp. Safetensors only.

Official tiny ladder only. Known SHA for Qwen2.5-0.5B-Instruct `model.safetensors`:
`fdf756fa7fcbe7404d5c60e26bff1a0c8b8aa1f72ced49e7dd0210fe288fb7fe` (988097824 bytes).
Do not invent hashes. Do not pull 27B/70B. Do not use GPU.

In [ ]:
ALLOW = {
    "Qwen/Qwen2.5-0.5B-Instruct": {
        "file": "model.safetensors",
        "sha256": "fdf756fa7fcbe7404d5c60e26bff1a0c8b8aa1f72ced49e7dd0210fe288fb7fe",
        "bytes": 988097824,
    },
    "Qwen/Qwen2.5-1.5B-Instruct": {
        "file": "model.safetensors",
        "sha256": "dd924a11b4c220f385b51ffa522daea7c9f3d850e31b162bb5661df483c6d3ee",
        "bytes": None,  # Will record actual size on first run
    },
    "Qwen/Qwen3-0.6B": {"file": "model.safetensors", "sha256": None, "bytes": None},
}
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
assert MODEL_ID in ALLOW
CACHE = "/tmp/modelscope_cache"
print("target", MODEL_ID, "scratch", CACHE)

In [ ]:
import hashlib, json, os, sys
from pathlib import Path

def sha256(path):
    h = hashlib.sha256(); n = 0
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk); n += len(chunk)
    return h.hexdigest(), n

try:
    from modelscope import snapshot_download
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "modelscope"])
    from modelscope import snapshot_download

model_dir = snapshot_download(
    MODEL_ID, cache_dir=CACHE,
    ignore_file_patterns=[r".*\.bin$", r".*\.msgpack$", r".*\.h5$", r".*\.ot$", r".*flax.*"],
)
spec = ALLOW[MODEL_ID]
target = os.path.join(model_dir, spec["file"])
rec = {"model_id": MODEL_ID, "dir": model_dir, "ok": False, "state": "incomplete"}
if os.path.exists(target):
    digest, size = sha256(target)
    rec.update({"file": spec["file"], "bytes": size, "sha256": digest})
    if spec["sha256"]:
        if digest != spec["sha256"] or (spec["bytes"] and size != spec["bytes"]):
            rec["error"] = "sha256/size mismatch — refuse"
            rec["state"] = "rejected"
        else:
            rec["ok"] = True
            rec["state"] = "pinned"
    else:
        rec["state"] = "observed-unpinned"
        rec["note"] = "hash recorded but NOT trusted. Do not promote."
        rec["ok"] = False
else:
    rec["error"] = "safetensors missing"
# complete-artifact extras
need = ["config.json", "tokenizer.json", "tokenizer_config.json"]
rec["extras"] = {n: os.path.exists(os.path.join(model_dir, n)) for n in need}
print(json.dumps(rec, indent=2))
open("/kaggle/working/download_manifest.json", "w").write(json.dumps(rec, indent=2))
assert rec["ok"], rec.get("error") or rec.get("state")

